In [23]:
# Libraries
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from scipy.stats import kruskal
from scipy.stats import spearmanr

In [24]:
games = pd.read_csv("../data/processed/game_lifecycle_classified.csv")
games

,app_id,title,launch_price,reviews_total,review_score,tags,baseline_players,relative_6m,relative_12m,relative_24m,early_trend_slope,month_6_vs_month_1,early_volatility,early_peak_to_average,lifecycle_category
0,730,Counter-Strike: Global Offensive,14.99,7382695,0.88,"FPS, Shooter, Multiplayer, Competitive, Action...",13624.776667,1.210952,1.813688,9.060906,0.046684,1.085745,0.158598,1.177936,Growing
1,2870,X Rebirth,49.99,5102,0.41,"Simulation, Space, Sci fi, Action, Space Sim, ...",617.486667,0.594242,0.605125,0.315354,-0.176420,0.467198,0.602581,2.149940,Declining
2,4920,Natural Selection 2,9.99,8198,0.85,"Multiplayer, Strategy, FPS, Team Based, Action...",2238.970000,0.493038,0.330320,0.165360,-0.179528,0.324727,0.387130,1.605843,Declining
3,8870,BioShock Infinite,59.99,98966,0.93,"FPS, Story Rich, Action, Singleplayer, Steampu...",3600.803333,0.422417,0.548959,0.295640,-0.264835,0.173542,0.829654,2.649201,Declining
4,10270,Disciples III: Reincarnation,29.99,1066,0.69,"RPG, Strategy, Turn Based Strategy, Turn Based...",80.743333,0.653057,0.668868,0.615572,-0.146955,0.398816,0.375316,1.736611,Sustained
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2537,799640,Dungeon Munchies,19.99,5597,0.97,"Indie, Action, Pixel Graphics, Adventure, RPG,...",54.923333,0.502033,0.251684,0.152212,-0.197041,0.312445,0.771824,2.533049,Declining
2538,800270,Terraforming Mars,24.99,4029,0.70,"Strategy, Board Game, Space, Tabletop, Turn Ba...",165.506667,1.013252,1.297269,2.531741,-0.062728,0.719030,0.180023,1.346088,Growing
2539,801480,Agent A: A puzzle in disguise,19.99,3864,0.95,"Logic, Point & Click, Hidden Object, Explorati...",11.000000,0.916970,0.942727,1.668182,-0.022287,0.465969,0.427646,1.639720,Growing
2540,801550,VAIL VR,19.99,694,0.76,"Early Access, VR, Shooter, Action, FPS, Multip...",44.466667,0.627511,0.501649,0.468966,-0.103055,0.531856,0.197713,1.197652,Declining


## Determine what characteristics distinguish games with different lifecycle outcomes

### Reviews

In [25]:
features = [
    "launch_price",
    "reviews_total",
    "review_score",
    "baseline_players"
]

games[features].describe()

,launch_price,reviews_total,review_score,baseline_players
count,2542.000000,2.542000e+03,2542.000000,2.542000e+03
mean,22.169300,1.727980e+04,0.815946,1.278295e+03
std,14.578402,1.610744e+05,0.126156,2.839802e+04
min,0.990000,1.000000e+01,0.130000,5.006667e+00
25%,11.990000,7.815000e+02,0.760000,1.702333e+01
50%,19.990000,2.334500e+03,0.840000,5.784333e+01
75%,29.990000,7.825000e+03,0.910000,2.607800e+02
max,179.990000,7.382695e+06,0.990000,1.423770e+06


In [26]:
order = ["Declining", "Sustained", "Growing"]

feature_summary = (games.groupby("lifecycle_category")[features].median().reindex(order))

feature_summary

# Games with better long-term retention tend to have substantially more total reviews
# Declining games have review score of 0.83 vs Sustained and Growing games have 0.88. Player reception may be associated with long-term retention.

,launch_price,reviews_total,review_score,baseline_players
lifecycle_category,,,,
Declining,19.99,1744.0,0.82,66.386667
Sustained,19.99,2700.5,0.86,58.676667
Growing,19.99,3551.0,0.88,32.910000


In [27]:
for feature in features:

    groups = [
        games.loc[games["lifecycle_category"] == category, feature].dropna()
        for category in order
    ]

    H, p = kruskal(*groups)

    print(f"{feature}: H={H:.2f}, p={p:.4g}")

launch_price: H=6.67, p=0.03564
reviews_total: H=100.58, p=1.442e-22
review_score: H=133.44, p=1.059e-29
baseline_players: H=28.29, p=7.191e-07


In [28]:
for feature in features:

    subset = games[[feature, "relative_24m"]].dropna()

    rho, p = spearmanr(subset[feature], subset["relative_24m"])

    print(
        f"{feature}: "
        f"rho={rho:.3f}, p={p:.4g}"
    )

launch_price: rho=0.008, p=0.6889
reviews_total: rho=0.232, p=1.974e-32
review_score: rho=0.277, p=5.012e-46
baseline_players: rho=-0.089, p=6.258e-06


Review score and review count have strongest positive association with 24-month retention. 

Baseline player count have a weak negative association with relative 24-month retention.

Launch price showed no significant association with long-term retention.

### Is multiplayer status associated with a game's 24-month lifecycle outcome?

H0​: Multiplayer status and lifecycle category are independent

H1: Multiplayer status and lifecycle category are associated

In [29]:
games["is_multiplayer"] = (games["tags"].str.contains("Multiplayer", case=False, na=False))

In [30]:
table = pd.crosstab(games["is_multiplayer"], games["lifecycle_category"])

table

lifecycle_category,Declining,Growing,Sustained
is_multiplayer,,,
False,787,250,421
True,510,233,341


In [31]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square statistic: 13.420671601930938
P-value: 0.0012182549647141159
Degrees of freedom: 2


In [32]:
pd.crosstab(
    games["is_multiplayer"],
    games["lifecycle_category"],
    normalize="index"
) * 100

lifecycle_category,Declining,Growing,Sustained
is_multiplayer,,,
False,53.978052,17.146776,28.875171
True,47.047970,21.494465,31.457565


0.00067 < 0.05

Therefore, reject null hypothesis.

Therefore, lifecycle outcome was significantly associated with multiplayer status

In [33]:
games.to_csv("../data/processed/game_lifecycle_analyzed.csv", index=False)